# Image Denoising

An in-progress comparison of spatial Gaussian smoothing and a learned convolutional denoiser. The paired training images and filtering examples are stored in `data/`; this notebook reads them locally and does not download datasets.

The learned model, training loop, and comparison are still being developed.


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

REPO_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data").is_dir()), Path.cwd())
DATA_ROOT = REPO_ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = DEVICE
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
from PIL import Image
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from skimage.metrics import peak_signal_noise_ratio
from tqdm.auto import tqdm


def plot_images_with_text(image_groups, titles, texts=None):
    row_count = max((len(group) for group in image_groups), default=0)
    figure, axes = plt.subplots(row_count, len(image_groups), figsize=(4 * len(image_groups), 3 * row_count), squeeze=False)
    for column, group in enumerate(image_groups):
        for row, image in enumerate(group):
            if torch.is_tensor(image):
                image = image.detach().cpu().squeeze().numpy()
                if image.ndim == 3 and image.shape[0] in (1, 3):
                    image = np.moveaxis(image, 0, -1)
            axes[row, column].imshow(image, cmap="gray" if image.ndim == 2 else None)
            axes[row, column].axis("off")
            if row == 0:
                axes[row, column].set_title(titles[column])
            if texts and row < len(texts):
                axes[row, column].set_xlabel(texts[row])
    plt.tight_layout()

## Visualization helpers


In [ ]:
def plot_images_horizontal(imgs):
    num_imgs = len(imgs)
    f = plt.figure(figsize=(40, 15))
    for i in range(num_imgs):
        f.add_subplot(1, num_imgs, i+1)
        plt.imshow(imgs[i])
    plt.show()

## Gaussian baseline

A fixed spatial filter provides a simple reference before the learned model.


In [ ]:
def gkern(l=5, sig=1.):

    return out_kernel

def denoise_gauss(image):

    return denoised_image

In [ ]:
dataset_path = str(DATA_ROOT)
noisy_image_filename = os.path.join(dataset_path, "Filtering/noisy_image.png")

In [ ]:
noisy_image = np.asarray(Image.open(noisy_image_filename).convert('RGB'))
denoised_image = denoise_gauss(noisy_image)
plot_images_horizontal([noisy_image, denoised_image])

## Learned denoiser

The CNN is trained on paired noisy and clean image patches.


In [ ]:
class DenoisingDB(Dataset):

    def __init__(self, input_imgs_path, cleaned_imgs_path):
        super().__init__()

    def __len__(self):

        return length

    def __getitem__(self, idx):

        return (input_img, cleaned_img)


In [ ]:
class DenoisingCNN(nn.Module):
    def __init__(self):
        super(DenoisingCNN, self).__init__()

    def forward(self, x):

        return x

# Create an instance of the network
model = DenoisingCNN()


In [ ]:
def loss_function(prediction, target):
    """
    Calculate the Mean Squared Error (MSE) loss between the prediction and the target.

    Arguments:
    prediction : torch.Tensor
        The predicted image.
    target : torch.Tensor
        The target image for reconstruction.

    Returns:
    loss : torch.Tensor
        The computed loss.
    """

    return loss


### Training data


In [ ]:
dataset_path = str(DATA_ROOT)
input_imgs_path = os.path.join(dataset_path, "Denoising/input_noisy_images/")
cleaned_imgs_path = os.path.join(dataset_path, "Denoising/target_clean_images/")

### Training loop


### Checkpointing and inspection


In [ ]:
if not os.path.exists("models/"):
  os.makedirs("./models/")
torch.save({"denoiser":denoiser}, "models/denoiser.pth")

In [ ]:
state_dict = torch.load("models/denoiser.pth", weights_only=False)
denoiser = state_dict["denoiser"].to(device)

## Comparison

The image selection, model training, and quantitative comparison remain open work.


In [ ]:
input_imgs, gt_imgs, output_imgs = [], [], []

titles = ['Input', 'GroundTruth', 'Output']
plot_images_with_text([input_imgs, gt_imgs, output_imgs],titles,texts=None)


In [ ]:
dataset = DenoisingDB(input_imgs_path,cleaned_imgs_path)

images = [dataset[i][0].to(device) for i in range(5)]
targets = [dataset[i][1].to(device) for i in range(5)]
images_ae = []
images_gauss = []


In [ ]:
imgs_plot = [ el.detach().squeeze().cpu().numpy() for el in images]
tgts_plot = [ el.detach().squeeze().cpu().numpy() for el in targets]

In [ ]:
psnrs = [ 0.0 for el in targets ]
for i in range(len(tgts_plot)):
    psnr_ae    = peak_signal_noise_ratio(tgts_plot[i], images_ae[i])
    pnsr_gauss = peak_signal_noise_ratio(tgts_plot[i], images_gauss[i])
    psnrs[i] = ['', psnr_ae, pnsr_gauss, '']

img = [imgs_plot, images_ae, images_gauss, tgts_plot]
titles = ['Input', 'AE', 'Gaussian', 'Targets']
texts = psnrs
plot_images_with_text(img, titles, texts)
plt.show()